# Lab 5, Module 1: Prompting Foundations

### *Tying Embedding to LLM *

**Estimated time: 15-20 minutes**

	-Try prompting in ChatGPT/Claude
	-Zero-shot, few-shot, chain-of-thought
	-Hallucination demo
	-Libraries: none


In [ ]:
# This cell:
#   • Loads a small, free word embedding model (GloVe)
#   
#
# NOTE: The first time you run this, it will download the model (~70MB),
#       which may take up to a minute in Colab.

import numpy as np
import matplotlib.pyplot as plt

try:
    import gensim.downloader as api
except ImportError:
    !pip install -q gensim
    import gensim.downloader as api

from sklearn.metrics.pairwise import cosine_similarity

### Loading a set of pre-embedded words

The gensim downloader will load a large set of words that already have been embedded into an abstract vector representation.  

In [ ]:
# ============================================================
#  Module 1 — Activity 0: Word-level Embedding
#  DATA 1010 – Artificial Intelligence in Action
# ============================================================



# -----------------------------
# 1. Load a pre-trained word embedding model
# -----------------------------
# Check if the model is already loaded to avoid reloading
if 'w2v' not in locals():
    print("Loading GloVe word vectors (glove-wiki-gigaword-50)...")
    w2v = api.load("glove-wiki-gigaword-50")  # 50-dimensional GloVe
    print("Model loaded!")
    print(f"Vocabulary size: {len(w2v.index_to_key):,} words")
    print(f"Vector dimension: {w2v.vector_size} dimensions\n")
else:
    print("GloVe word vectors (w2v) already loaded.\n")



### Let's take a look at a few of the words and how they are represented

In [ ]:

np.set_printoptions(precision=3,linewidth=60)
w = "galaxy"
print(f"word: {w:20s} ")
print(f"Length of embedding: {len(w2v[w])}")
print(f"embedding: \n{ w2v[w]}\n")

### Understanding the dimensions

We can the vectors of a bunch of words an see how this looks on a 2d chart.  Each line represents the embedding of a different word.  You can't really tell a lot by looking at the graph, but notice the peak around parameter 30.   That value seems to be higher for "person" and "table" than for "galaxy" and "atom".


In [ ]:
science_words =  ["galaxy", "person", "table", "atom"]
x = list(range(50))
np.set_printoptions(precision=3,linewidth=60)
for w in science_words:
    print(f"word: {w:20s} ")
    y = w2v[w]
    plt.plot(x,y, label=w)
    
plt.annotate('Peak', xy=(30, 2), xytext=(25, 2.5),
             arrowprops=dict(facecolor='black', shrink=0.05),
             ha='right', va='bottom')
plt.legend ( )
plt.xlabel( "Dimension")
plt.ylabel("value")

### Let's explroe this parameter a bit.

1) Take a bunch of science words - find all their vectors and average them
2) Take a bunch of non-science words - find all their vectors and averge them
3) Subtract the non-science from the science, and plot the results

In [ ]:
science_words = science_words + ["galaxy", "atom", "molecule", "quantum","telescope", "cell", "nucleus", "research", "experiment"]
nonscience_words = ["cat", "dog", "pizza", "music", "tree", "happy", "running", "house"]


science_average = np.zeros(50)
science_ct = 0
for w in science_words:
  word_vector = w2v[w]
  science_average = science_average + np.array(word_vector)
  science_ct = science_ct + 1
science_average = science_average / science_ct
  
nonscience_average = np.zeros(50)
nonscience_ct = 0
for w in nonscience_words:
  word_vector = w2v[w]
  nonscience_average = nonscience_average + np.array(word_vector)
  nonscience_ct = nonscience_ct + 1
nonscience_average = nonscience_average / nonscience_ct
  
science_displacement = science_average - nonscience_average
x = np.array(list(range(50)))

plt.plot(x,science_displacement,"*")
plt.xlabel("embedding parameter")
plt.ylabel("displacement from non-science words to science words")


### What happened?

There are clear differences between science words and non-science words.  However, the biggest displayment seems to be parameter 33 where the diference science words average about 1.7 lower than non-science words.

Let's explore this with some other science words and see what happens with parameter 33 and a few other random parameters.

In [ ]:
new_science_words = ["economics", "microbiology","zoology","biochemistry","oceanography","science","chemistry","physics","biology","meteorology","geology","mathematics","astronomy","astrophysics"]

p1 = np.random.randint(32)
p2 = np.random.randint(16) + 34

print(f"  word                  P: 33     P: {p1:02d}    P:{p2:02d}")
wlist = []
for w in new_science_words:
  wa = w2v[w]
  wlist.append(wa)
  print(f"{w:20s} {wa[33]-nonscience_average[33]:>8.3f}  " \
    + f"{wa[p1]-nonscience_average[p1]:>8.3f} " \
    + f"{wa[p2]-nonscience_average[p2]:>8.3f} ")

### Is parameter 33 how "science-ey" a word is?

No.  

However, you can see from above, parameter 33 seems to be a fairly reliable marker of how "science-ey" a word is. Physics is very "science-ey" (P33 = -2.626) and economics is moderately "science-ey" (P33=-1.520).   However, the actual meaning of parameter 33 is more complex.  It is just aligned somewhat with the concept we call science.  It is NOT actually this, but something that machine has figured out as an important concept in understanding the relationships between words.


# **Module 1 — Activity 3: Vector Arithmetic & Analogies**

# # **Module 2 — Activity 3: Vector Arithmetic & Analogies**
 In this activity, you will explore how meaning can be represented as **vectors**, and how
 simple arithmetic on these vectors can capture relationships between words.

 We will use analogies of the form:

 **A − B + C  ≈  ?**

 where:
 - **A** is a *changed* form (plural, past, comparative, capital, etc.)
 - **B** is the *base* form (singular, present, base adjective, country, etc.)
 - **C** is a *new base* you want to transform.

Examples of the intended pattern:
 - children − child + person  ≈  people  
 - walked − walk + swim       ≈  swam  
 - smaller − small + big      ≈  bigger  
 - paris − france + italy     ≈  rome  


Below we will use the small embedding model (GloVe-50) and test:

 - Country ↔ Capital  
 - Comparatives (big → bigger)  
 - Verb tenses (walk → walked)  
 - Pluralization  
 - Family roles  



In [ ]:
# ============================================================
#  Module 2 — Activity 3: Vector Arithmetic & Analogies
#  DATA 1010 – Artificial Intelligence in Action
# ============================================================

# This cell:
#   • Demonstrates classic analogies like: king - man + woman ≈ queen
#   • Lets you try your own word analogies
#

# -----------------------------
# 1. Load a pre-trained word embedding model
# -----------------------------
# Check if the model is already loaded to avoid reloading
if 'w2v' not in locals():
    print("Make sure to execute the top of the notebook before trying this cell.")
    exit()
else:
    print("GloVe word vectors (w2v) already loaded.\n")

# -----------------------------
# 2. Helper function: show analogy
# -----------------------------
def show_analogy(word_a, word_b, word_c, topn=5):
    """
    Compute:  word_a - word_b + word_c  ≈  ?
    and print the top similar words.
    """
    print("===============================================")
    print(f"Analogy:  {word_a}  -  {word_b}  +  {word_c}  ≈  ?")
    print("===============================================")

    # Check vocabulary
    for w in [word_a, word_b, word_c]:
        if w not in w2v:
            print(f"  • The word '{w}' is not in the model vocabulary.")
            return

    # Vector arithmetic
    result_vec = w2v[word_a] - w2v[word_b] + w2v[word_c]

    # Find most similar words to result_vec
    sims = w2v.similar_by_vector(result_vec, topn=topn)

    for rank, (word, score) in enumerate(sims, start=1):
        print(f"{rank}. {word:15s}  (cosine similarity: {score:.4f})")

    print("\n")




# -----------------------------
# 3. Reliable Example Analogies
# -----------------------------
print("### Reliable Analogy Examples ###\n")

# Capital–Country (A = capital, B = country, C = new country)
show_analogy("paris",   "france",  "italy")     # → rome
show_analogy("berlin",  "germany", "spain")     # → madrid

# Comparatives (A = comparative, B = base adj, C = new base adj)
show_analogy("smaller", "small",   "big")       # → bigger
show_analogy("colder",  "cold",    "warm")      # → warmer

# Verb tenses (A = past, B = present, C = new present)
show_analogy("walked",  "walk",    "swim")      # → swam
show_analogy("made",    "make",    "think")     # → thought

# Plurals (A = plural, B = singular, C = new singular)
show_analogy("children","child",   "person")    # → people
show_analogy("dogs",    "dog",     "cat")       # → cats

# Family roles (A = female, B = male, C = new male)
show_analogy("aunt",    "uncle",   "brother")   # → sister
show_analogy("mother",  "father",  "son")       # → daughter



### Not all relationships work well as vector analogies.

This model is particularly good at:

capital–country
singular–plural
present–past verbs
base–comparative adjectives
It is not very good at:

animal → sound
“vibes” (e.g., cozy, spooky)
pop culture or memes
So if your analogy fails, it doesn’t mean you did it wrong – it often means the relationship isn’t represented as a simple line in this embedding space.

In [ ]:
# -----------------------------
# 4. Try your own analogy
# -----------------------------
print("Now try your own analogy!")
print("Enter three words to compute:  A - B + C  ≈  ?")
print("Example:  A = king, B = man, C = woman\n")

word_a = input("Enter word A: ").strip().lower()
word_b = input("Enter word B: ").strip().lower()
word_c = input("Enter word C: ").strip().lower()

show_analogy(word_a, word_b, word_c)
